# Week 7 Exercise — Open-Source Sentiment Classifier

This project demonstrates how to fine-tune an open-source language model
using QLoRA for sentiment classification.

A subset of the IMDB dataset from Hugging Face is used to train a Llama-3.2-3B
model to classify reviews as positive or negative.

The workflow includes:
1. Loading and preparing the dataset
2. Formatting prompts for supervised training
3. Fine-tuning an open-source model with QLoRA
4. Evaluating model performance

Training is designed to run on a GPU environment such as Google Colab.


In [ ]:
!pip install -q transformers datasets peft bitsandbytes accelerate trl scikit-learn

In [ ]:
import torch
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

from sklearn.metrics import accuracy_score

In [ ]:
dataset = load_dataset("imdb")

train_data = dataset["train"].select(range(2000))
test_data = dataset["test"].select(range(500))

print("Training samples:", len(train_data))
print("Test samples:", len(test_data))

In [ ]:
def label_to_sentiment(label):
    return "positive" if label == 1 else "negative"

In [ ]:
def format_prompt(example):

    review = example["text"]
    sentiment = label_to_sentiment(example["label"])

    text = f"""
    Classify the sentiment of this movie review.

    Review:
    {review}

    Answer:
    {sentiment}
    """

    return {"text": text}

train_dataset = train_data.map(format_prompt)
test_dataset = test_data.map(format_prompt)

In [ ]:
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj"]
)

In [ ]:
training_config = SFTConfig(
    output_dir="sentiment_model",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    logging_steps=20,
    max_length=256,
    dataset_text_field="text",
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=training_config
)

In [ ]:
trainer.train()

In [ ]:
def predict_sentiment(review):

    prompt = f"""
    Classify the sentiment of this movie review.

    Review:
    {review}

    Answer:
    """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=5
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return result

In [ ]:
example = test_data[0]["text"]

print(example)
print("\nPrediction:")
print(predict_sentiment(example))

In [ ]:
predictions = []
labels = []

for item in test_data.select(range(50)):

    review = item["text"]
    true_label = label_to_sentiment(item["label"])

    prediction = predict_sentiment(review)

    predictions.append(prediction)
    labels.append(true_label)

accuracy = accuracy_score(labels, predictions)

print("Accuracy:", accuracy)

In [ ]:
import gradio as gr

def classify_review(review):
    return predict_sentiment(review)

demo = gr.Interface(
    fn=classify_review,
    inputs=gr.Textbox(label="Product Review"),
    outputs=gr.Textbox(label="Sentiment"),
    title="Product Review Sentiment Classifier"
)

demo.launch()